# 103 - Building an Option Chain

This notebook shows how to construct a complete option chain for SPY from
scratch:

1. List expirations and pick the nearest monthly
2. Fetch all strikes
3. Pull snapshot quotes for calls and puts
4. Assemble a unified option chain DataFrame
5. Visualize open interest across strikes

For the implied volatility smile and the full Greeks surface, see
[104 - Greeks and Volatility Surface](104_greeks_surface.ipynb), which
reads ThetaData's served Greeks straight off the snapshot rows.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

from thetadatadx import Credentials, Config, Client

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

In [ ]:
creds = Credentials.from_file("creds.txt")
client = Client(creds, Config.production())

## 1. List All Expirations for SPY

Expirations come back as ISO date strings (`"2024-03-15"`). Keep only
those in the future.

In [ ]:
expirations = client.historical.option_list_expirations("SPY").to_list()
today = datetime.now().date().isoformat()
expirations = [e for e in expirations if e > today]
print(f"Upcoming expirations: {len(expirations)}")
print(f"First 10: {expirations[:10]}")

## 2. Pick the Nearest Monthly Expiration

SPY has daily (0DTE), weekly, and monthly expirations. Monthly
expirations fall on the third Friday of each month. We pick the first
expiration that is on a Friday, in the 15th-21st day window, and at least
14 days out.

In [ ]:
today_dt = datetime.now()

monthly_exp = None
for exp_str in expirations:
    exp_date = datetime.strptime(exp_str, "%Y-%m-%d")
    days_out = (exp_date - today_dt).days
    # Monthly expirations are on Fridays, at least 14 days out
    if days_out >= 14 and exp_date.weekday() == 4:  # Friday
        # Third-Friday heuristic: day-of-month between 15 and 21
        if 15 <= exp_date.day <= 21:
            monthly_exp = exp_str
            break

# Fallback: first expiration at least 20 days out
if monthly_exp is None:
    for exp_str in expirations:
        exp_date = datetime.strptime(exp_str, "%Y-%m-%d")
        if (exp_date - today_dt).days >= 20:
            monthly_exp = exp_str
            break

exp_dt = datetime.strptime(monthly_exp, "%Y-%m-%d")
dte = (exp_dt - today_dt).days
print(f"Selected expiration: {monthly_exp} ({dte} DTE)")

## 3. Fetch All Strikes for That Expiration

Strikes come back as dollar-value strings (`"540"`). We also grab the
current spot from a stock OHLC snapshot - snapshot endpoints return a
list of typed rows, so we read `.close` off the first row.

In [ ]:
all_strikes = client.historical.option_list_strikes("SPY", monthly_exp).to_list()
print(f"Total strikes: {len(all_strikes)}")

# Current spot price from a stock OHLC snapshot (one row per symbol).
spot_price = client.historical.stock_snapshot_ohlc(["SPY"])[0].close
print(f"SPY spot: ${spot_price:.2f}")

# Sort strikes by dollar value, then keep those within +/-10% of spot.
strike_pairs = sorted(((s, float(s)) for s in all_strikes), key=lambda x: x[1])
filtered = [s for s, v in strike_pairs if spot_price * 0.90 <= v <= spot_price * 1.10]
print(f"Strikes within +/-10%: {len(filtered)}")

## 4. Fetch Snapshot Quotes for Calls and Puts

For each strike, fetch the current NBBO quote for the call and the put.
Option snapshot methods are keyword-only for `expiration` / `strike` /
`right`, take the strike as a dollar string, and return a list of typed
rows.

In [ ]:
chain_rows = []

for strike_str in filtered:
    strike_val = float(strike_str)

    try:
        call_q = client.historical.option_snapshot_quote(
            "SPY", expiration=monthly_exp, strike=strike_str, right="C")
        put_q = client.historical.option_snapshot_quote(
            "SPY", expiration=monthly_exp, strike=strike_str, right="P")
    except Exception:
        continue

    row = {"strike": strike_val}

    if call_q:
        c = call_q[0]
        row["call_bid"], row["call_ask"] = c.bid, c.ask
        row["call_mid"] = (c.bid + c.ask) / 2
    else:
        row["call_bid"] = row["call_ask"] = row["call_mid"] = np.nan

    if put_q:
        p = put_q[0]
        row["put_bid"], row["put_ask"] = p.bid, p.ask
        row["put_mid"] = (p.bid + p.ask) / 2
    else:
        row["put_bid"] = row["put_ask"] = row["put_mid"] = np.nan

    chain_rows.append(row)

chain = pd.DataFrame(chain_rows).sort_values("strike").reset_index(drop=True)
print(f"Option chain: {len(chain)} strikes")
chain.head(10)

## 5. Build the Complete Option Chain DataFrame

A classic chain layout: calls on the left, strike in the center, puts on
the right, centered on the ATM strike.

In [ ]:
display_cols = ["call_bid", "call_ask", "call_mid", "strike",
                "put_bid", "put_ask", "put_mid"]
chain_display = chain[display_cols].copy()

# Highlight the ATM strike
atm_idx = (chain["strike"] - spot_price).abs().idxmin()
atm_strike = chain.loc[atm_idx, "strike"]
print(f"ATM strike: ${atm_strike:.0f}  (spot: ${spot_price:.2f})\n")

# Show 10 strikes around ATM
start = max(0, atm_idx - 5)
end = min(len(chain), atm_idx + 6)
chain_display.iloc[start:end]

## 6. Open Interest Across Strikes

Open interest is published once per day (updated overnight). The snapshot
returns a list of typed rows; read `.open_interest` off the first row.

In [ ]:
call_oi_list = []
put_oi_list = []

for strike_str in filtered:
    try:
        call_oi = client.historical.option_snapshot_open_interest(
            "SPY", expiration=monthly_exp, strike=strike_str, right="C")
        put_oi = client.historical.option_snapshot_open_interest(
            "SPY", expiration=monthly_exp, strike=strike_str, right="P")
    except Exception:
        call_oi_list.append(0)
        put_oi_list.append(0)
        continue

    call_oi_list.append(call_oi[0].open_interest if call_oi else 0)
    put_oi_list.append(put_oi[0].open_interest if put_oi else 0)

chain["call_oi"] = call_oi_list
chain["put_oi"] = put_oi_list

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

width = (chain["strike"].diff().median()) * 0.35 if len(chain) > 1 else 1
ax.bar(chain["strike"] - width / 2, chain["call_oi"] / 1000, width=width,
       color="#1f77b4", alpha=0.8, label="Call OI")
ax.bar(chain["strike"] + width / 2, chain["put_oi"] / 1000, width=width,
       color="#d62728", alpha=0.8, label="Put OI")

ax.axvline(x=spot_price, color="gray", linestyle="--", linewidth=1, alpha=0.7,
           label=f"Spot ${spot_price:.0f}")

ax.set_xlabel("Strike ($)")
ax.set_ylabel("Open Interest (thousands)")
ax.set_title(f"SPY Open Interest by Strike - {monthly_exp}")
ax.legend()

plt.tight_layout()
plt.show()

---

**Next:** [104 - Greeks and Volatility Surface](104_greeks_surface.ipynb)